# Assignment 10 — Fine-Tune & Deploy a Domain-Specific LLM

**Module 8 · Fine-Tuning & Optimization**

Fine-tune **Qwen2.5-1.5B** on a synthetic FastAPI instruction dataset using **LoRA + Unsloth**,
quantize to **GGUF (Q4_K_M)**, and run the result locally through **Ollama**.

Full pipeline:

| Step | Where | Output |
|------|-------|--------|
| 1. Generate synthetic dataset | Local (Ollama) | `fastapi_dataset.json` |
| 2. LoRA fine-tune | **Colab GPU** | LoRA adapter |
| 3. Merge + quantize | **Colab GPU** | `.gguf` (Q4_K_M) |
| 4. Register + compare | Local (Ollama) | fine-tuned model |

**This notebook must run on Google Colab with a GPU** — Unsloth requires CUDA and will not run
on a Mac. Set `Runtime → Change runtime type → T4 GPU`.

Step 1 runs locally first:

```bash
python Assignment10_dataset.py     # writes fastapi_dataset.json
```

Then upload `fastapi_dataset.json` to this Colab session.

### Domain: the FastAPI web framework

Chosen because the base model has seen enough FastAPI for the teacher model to produce correct
training data, answers are objectively checkable (the code runs or it does not), and the same
domain is reused in Assignment 11.

In [ ]:
# ============================================================
# 1. Install Unsloth (Colab)
# ============================================================

# Unsloth pins compatible versions of torch, transformers, trl and peft.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
# ============================================================
# 2. Load the Base Model in 4-bit
# ============================================================

from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
BASE_MODEL = "unsloth/Qwen2.5-1.5B-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,            # None = auto-detect (bfloat16 on Ampere, float16 on T4)
    load_in_4bit=True,     # QLoRA: 4-bit base weights, LoRA adapters in 16-bit
)

print("Base model loaded:", BASE_MODEL)

## 3. Baseline Answers — Before Fine-Tuning

Capture the base model's answers to the evaluation questions **now**, while the adapter is not
yet attached. Comparing against a remembered impression instead of recorded output is how
people convince themselves fine-tuning worked when it did not.

In [ ]:
# ============================================================
# 3. Record Baseline Outputs
# ============================================================

EVAL_QUESTIONS = [
    "How do I add a query parameter with a default value in FastAPI?",
    "How do I return a 404 error from a FastAPI endpoint?",
    "How do I use dependency injection in FastAPI?",
    "How do I stream a response from FastAPI using Server-Sent Events?",
    "How do I upload a file in FastAPI?",
]

ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""


def generate(question, max_new_tokens=220):
    """Generate one answer from the model currently in memory."""

    FastLanguageModel.for_inference(model)

    inputs = tokenizer(
        [ALPACA_PROMPT.format(question, "", "")],
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.1,
        do_sample=False,
    )

    decoded = tokenizer.batch_decode(outputs)[0]

    return decoded.split("### Response:")[-1].strip()


baseline_answers = {}

for question in EVAL_QUESTIONS:
    baseline_answers[question] = generate(question)
    print("=" * 70)
    print("Q:", question)
    print(baseline_answers[question][:400])

In [ ]:
# ============================================================
# 4. Attach LoRA Adapters (Attention Layers Only)
# ============================================================

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                       # LoRA rank required by the assignment
    target_modules=[            # attention projections only
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,             # 0 is optimised in Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# ---- Verify trainable parameters are under 5% of total ----

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

percentage = 100 * trainable / total

print(f"Trainable parameters : {trainable:,}")
print(f"Total parameters     : {total:,}")
print(f"Trainable percentage : {percentage:.4f}%")
print(f"Under 5% requirement : {percentage < 5}")

In [ ]:
# ============================================================
# 5. Load and Format the Dataset
# ============================================================

import json

from datasets import Dataset

# Upload fastapi_dataset.json to the Colab session first
# (Files panel on the left, or files.upload() below).
#
# from google.colab import files
# files.upload()

with open("fastapi_dataset.json", "r", encoding="utf-8") as file:
    rows = json.load(file)

print("Loaded pairs:", len(rows))

EOS_TOKEN = tokenizer.eos_token


def format_row(row):
    """Render one pair into the Alpaca prompt, ending with EOS.

    Without EOS the model never learns to stop and generates until it hits
    max_new_tokens.
    """

    return ALPACA_PROMPT.format(
        row["instruction"],
        row.get("input", ""),
        row["output"],
    ) + EOS_TOKEN


dataset = Dataset.from_list(rows)

dataset = dataset.map(
    lambda row: {"text": format_row(row)},
)

print("\nFormatted example:\n")
print(dataset[0]["text"][:600])

In [ ]:
# ============================================================
# 6. Fine-Tune with SFTTrainer
# ============================================================

from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,      # effective batch size 8
        warmup_steps=5,
        max_steps=60,                       # assignment asks for 50-100
        learning_rate=2e-4,                 # assignment specified
        logging_steps=10,                   # log loss every 10 steps
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# Older TRL versions take tokenizer= and dataset_text_field= on SFTTrainer
# itself rather than on SFTConfig. If the call above raises a TypeError,
# move those two arguments up to SFTTrainer.

trainer_stats = trainer.train()

In [ ]:
# ============================================================
# 7. Training Summary
# ============================================================

print("Runtime (s):", round(trainer_stats.metrics["train_runtime"], 2))
print("Final training loss:", round(trainer_stats.metrics["train_loss"], 4))

print("\nLoss every 10 steps:")

for record in trainer.state.log_history:
    if "loss" in record:
        print(f"  step {record['step']:3d} | loss {record['loss']:.4f}")

In [ ]:
# ============================================================
# 8. Compare Answers - Base vs Fine-Tuned
# ============================================================

for question in EVAL_QUESTIONS:

    print("\n" + "=" * 78)
    print("QUESTION:", question)
    print("=" * 78)

    print("\n--- BASE MODEL (before fine-tuning) ---")
    print(baseline_answers[question][:400])

    print("\n--- FINE-TUNED MODEL ---")
    print(generate(question)[:400])

In [ ]:
# ============================================================
# 9. Save the LoRA Adapter
# ============================================================

model.save_pretrained("fastapi_lora_adapter")
tokenizer.save_pretrained("fastapi_lora_adapter")

print("Adapter saved. Size on disk:")
!du -sh fastapi_lora_adapter

# The adapter alone is a few tens of MB - the base model is unchanged.
# This is the practical advantage of LoRA: you ship a small diff, not a model.

In [ ]:
# ============================================================
# 10. Merge and Export to GGUF (Q4_K_M)
# ============================================================

# Merges the LoRA adapter back into the base weights, converts to GGUF and
# quantizes in one call. This takes several minutes and needs disk space.

model.save_pretrained_gguf(
    "fastapi_model_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

!ls -lh fastapi_model_gguf/

In [ ]:
# ============================================================
# 11. Download the GGUF File
# ============================================================

import glob

gguf_files = glob.glob("fastapi_model_gguf/*.gguf")

print("GGUF files produced:", gguf_files)

# In Colab, download to your local machine:
#
# from google.colab import files
# files.download(gguf_files[0])
#
# The file is roughly 1 GB for a 1.5B model at Q4_K_M, so the browser download
# can be slow. Mounting Google Drive and copying there is usually faster.

## 12. Register with Ollama (run locally)

Copy the `.gguf` file next to `Assignment10_Modelfile`, then:

```bash
ollama create fastapi-expert -f Assignment10_Modelfile

ollama run fastapi-expert "How do I return a 404 error from a FastAPI endpoint?"
```

Compare base and fine-tuned side by side on the 5 evaluation questions:

```bash
python Assignment10_compare.py
```

## When to fine-tune vs RAG vs prompt engineering

The decision framework to walk through in the video:

| | Use it when | Avoid it when |
|---|---|---|
| **Prompt engineering** | The base model already knows the content; you need format or tone control. Zero cost, instant iteration. | The knowledge genuinely is not in the model. |
| **RAG** | Knowledge is large, changes often, or must be cited. Facts update without retraining. | You need a different *style*; retrieval cannot change how a model writes. |
| **Fine-tuning** | You need consistent format, tone or a task the model performs poorly. Also cuts prompt length, so it lowers per-call cost at scale. | Facts change frequently — every update means retraining. |

The distinction that matters: **fine-tuning teaches behaviour, RAG supplies facts.** A model
fine-tuned on FastAPI answers *in the style* of the training data, but it does not learn today's
FastAPI release notes. Those belong in RAG. Fine-tuning on frequently-changing facts bakes in
knowledge that silently goes stale, which is the most common and most expensive mistake here.

**Cost/quality trade-off in this assignment:** LoRA trains under 1% of parameters for a few
minutes on a free T4, and Q4_K_M quantization cuts the model to roughly a quarter of its
fp16 size at a small quality cost — which is what makes a 1.5B model practical to run locally.
The honest limitation is the dataset: ~60 synthetic pairs from a small teacher model is enough
to shift *style*, not to add real capability. A production run would use thousands of
human-reviewed pairs and a held-out eval set rather than 5 spot-check questions.